# 02 — Model Experiments

Trains and compares a few models using **`src` functions** (same code the
training pipeline uses), on a short horizon for speed. Illustrative, on sample
data. See `scripts/run_training_pipeline.py` for the full comparison + registry.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.express as px

from pearls_aqi.config import load_config
from pearls_aqi.storage import get_feature_store
from pearls_aqi.storage.base import FEATURES_GROUP
from pearls_aqi.features.engineering import select_feature_columns
from pearls_aqi.features.targets import build_direct_targets, target_columns, trainable_rows
from pearls_aqi.evaluation.splits import chronological_split
from pearls_aqi.evaluation.metrics import regression_metrics, metrics_by_horizon
from pearls_aqi.models import build_model

HORIZON = 24
cfg = load_config(); loc = cfg.active_location
df = get_feature_store(cfg).read_features(FEATURES_GROUP, city_id=loc.city_id)
df = trainable_rows(build_direct_targets(df, HORIZON), HORIZON)
tcols = target_columns(HORIZON)
fcols = select_feature_columns(df, tcols)
print(f"{len(df):,} rows, {len(fcols)} features, horizon {HORIZON}h")

In [ ]:
split = chronological_split(df, 0.7, 0.15, 0.15, gap_hours=HORIZON)
print(split.summary())
Xtr, ytr = split.train[fcols], split.train[tcols]
Xva, yva = split.validation[fcols], split.validation[tcols]

In [ ]:
rows = {}
fitted = {}
for name in ["persistence", "seasonal_naive", "ridge", "random_forest"]:
    m = build_model(name, HORIZON, cfg.training, tune=False).fit(Xtr, ytr)
    fitted[name] = m
    rows[name] = regression_metrics(yva.to_numpy(), m.predict(Xva))
pd.DataFrame(rows).T[["mae", "rmse", "r2"]].round(3).sort_values("mae")

In [ ]:
# Actual vs predicted (horizon 1) for the best linear model
best = "ridge"
pred = fitted[best].predict(Xva)[:, 0]
comp = pd.DataFrame({"actual": yva.iloc[:, 0].to_numpy(), "predicted": pred})
px.scatter(comp, x="actual", y="predicted", opacity=0.4,
           title=f"{best}: actual vs predicted AQI (t+1)").show()

In [ ]:
# Error-by-horizon degradation
mbh = metrics_by_horizon(yva.to_numpy(), fitted["random_forest"].predict(Xva))
px.line(mbh, x="horizon_hour", y="mae", markers=True,
        title="random_forest — MAE by forecast horizon").show()

**Takeaways (sample data).** Error grows with horizon (expected). The full
pipeline (`run_training_pipeline.py`) compares nine approaches, tunes linear/tree
models with time-series CV, selects on validation MAE, and registers + promotes
the winner. Re-run on production data for meaningful accuracy.